In [2]:
!pip install streamlit ultralytics -q

In [6]:
%%writefile app.py
import streamlit as st
from ultralytics import YOLO
from PIL import Image
import cv2

st.set_page_config(page_title="People & Object Detector", layout="centered")
st.title("Upload Image → Detect People Instantly")
st.caption("Free • No login • Works on phone")

@st.cache_resource
def load_model():
    return YOLO("yolo11n.pt")   # auto-downloads first time

model = load_model()

uploaded = st.file_uploader("Upload photo", type=["jpg","jpeg","png","webp"])

if uploaded:
    img = Image.open(uploaded)
    st.image(img, caption="Original", use_column_width=True)

    with st.spinner("Detecting..."):
        result = model(img, verbose=False)[0]

        people = sum(1 for c in result.boxes.cls if int(c) == 0)

        annotated = result.plot()
        annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

        st.image(annotated, caption=f"Found {people} people!", use_column_width=True)

        if people > 0:
            st.success(f"Detected {people} people! 🎉")
            if people >= 7:
                st.balloons()
        else:
            st.info("No people found — try a clearer photo")

Overwriting app.py


In [7]:
# This one always works — no ngrok, no localtunnel issues
!wget -q -O - https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 > cloudflared
!chmod +x cloudflared
!nohup ./cloudflared --url http://localhost:8501 &
!sleep 5
!cat nohup.out

nohup: appending output to 'nohup.out'
2025-12-04T16:02:06Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2025-12-04T16:02:06Z INF Requesting new quick Tunnel on trycloudflare.com...
2025-12-04T16:02:10Z INF +--------------------------------------------------------------------------------------------+
2025-12-04T16:02:10Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2025-12-04T16:02:10Z INF |  https://produc

In [8]:
!streamlit run app.py --server.port 8501 &>/dev/null &
import time, requests
time.sleep(8)
print("YOUR APP IS READY! Click this link →")
!curl -s http://localhost:4040/api/tunnels | python3 -c "import sys, json; print(json.load(sys.stdin)['tunnels'][0]['public_url'])"

YOUR APP IS READY! Click this link →
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/usr/lib/python3.12/json/__init__.py", line 293, in load
    return loads(fp.read(),
           ^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/json/decoder.py", line 338, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/json/decoder.py", line 356, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)
